[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/REPPL/Teaching/blob/main/MN5813/2026-27/Week/02b/Quest.ipynb)


<!-- gen:values.yml#weeks.02b.notebook-title|Quest 02b: The Subscription Churn Memo -->
# Week 2b Quest: The Subscription Churn Memo
<!-- /gen -->

*A streaming startup, a nervous board, and an [Associate](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-associate) draft with something quietly wrong in it.*

**About this [quest](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-quest).** It takes about 90 minutes. Everything you need is in this week's [Demonstration](./Demonstration.ipynb). Nothing needs the Demonstration's Stretch section. Work with your Associate throughout. Keep your best and worst exchanges for the [AI Use Record](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-ai-use-record) cell at the end.

## 📋 Stage 1: Briefing

*(about 10 minutes)*

> **From:** A. Partner, [Runnymede Analytics](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-runnymede-analytics)
> **To:** You ([Junior Analyst](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-analyst))
> **Re:** StreamShore: Churn numbers the board can trust
>
> New client: **StreamShore**, a streaming startup, eighteen months old. It has three subscription plans: Basic £5.99, Standard £9.99, and Premium £13.99. The board meets on Friday to decide which plan to push next quarter. Inside the company, the rumour is that *"Premium is our weakest plan"*. Nobody has checked.
>
> They have sent their full subscriber list. I need a short memo with two things: **How bad churn is, plan by plan**, and **how long a customer typically stays**. StreamShore's own Associate has already tried to calculate the second number. Look at its work before you trust it.
>
> One more thing. Their data has history: The founding members were kept in a spreadsheet before the billing system existed. Imported data has moods. Find out what kind of mood before you compute anything with money in it.
>
> A.P.

**The business question:** Which plans are losing customers fastest, and how long does a StreamShore customer stay?

**The deliverable:** Printed, verified numbers a memo could quote. You need a per-plan churn table and a customer-lifetime figure whose *definition you can defend out loud*.

**[Definition of Done](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-definition-of-done)**: Tick these yourself before claiming a [badge](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-badge):

- [ ] Every ✏️ prediction cell in [First look](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-first-look) filled in *before* you run its code cell
- [ ] Churn table printed: Subscribers, cancellations, and churn rate to one decimal place, per plan
- [ ] The three plan counts sum to 320 and the cancellations sum to the total printed by the data-setup cell
- [ ] The Associate's lifetime draft judged: Both fixes understood, one implemented, choice defended **in business terms** in writing
- [ ] The [Red Flags](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-red-flags) bug found, explained in business terms, and fixed by directing your Associate
- [ ] All three [Debrief](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-debrief) prompts answered, and the AI Use Record cell filled in
- [ ] A dated entry in your `QuestLog.md` with the badge you are claiming

**Badge criteria this week:**

- 🥉 **[Bronze](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-bronze)**: Churn count *and* churn rate per plan, computed with your Associate and verified (totals reconcile, plus one hand-check)
- 🥈 **[Silver](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-silver)**: Bronze, *plus* the customer-lifetime judgement: Both proposed fixes understood, one chosen, and the choice defended in writing in business terms
- 🥇 **[Gold](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-gold)**: Silver, *plus* an open analysis of your own design (for example, revenue retained by signup cohort) that adds one decision-relevant sentence to the memo

*Stage timings: Briefing 10 · First look 15 · Build 40 · Red Flags 15 · Debrief 10-15 minutes.*

### The data

Run the cell below first. It generates StreamShore's subscriber list with the module's fixed seed (`5813`), so every analyst sees the same data. Each subscriber is one dict (a set of labelled values) with these keys:

- `id`: Subscriber reference
- `plan`: `"Basic"`, `"Standard"`, or `"Premium"`
- `signup_month`: Months since launch (month 0 = January 2025)
- `cancelled_month`: The month they cancelled, or **`None` if still subscribed**
- A monthly fee field: The Partner's warning about imported data applies

The first month is a free trial. A subscriber who signs up and cancels in the same month pays for zero months.

In [ ]:
# --- Data setup: Run this cell first (runs anywhere, no download needed) ---
import numpy as np

rng = np.random.default_rng(5813)  # module seed: Do not change

# StreamShore launched in January 2025. Months are numbered from launch:
# month 0 = January 2025, so "today" (June 2026) is month 17.
TODAY = 17

PLAN_FEES = {"Basic": 5.99, "Standard": 9.99, "Premium": 13.99}
CHURN_RISK = {"Basic": 0.055, "Standard": 0.038, "Premium": 0.022}
PLAN_MIX = ["Basic"] * 24 + ["Standard"] * 54 + ["Premium"] * 22

subscribers = []

# App sign-ups: Joined at some point between launch and month 14.
for i in range(276):
    plan = PLAN_MIX[int(rng.integers(0, 100))]
    signup = int(rng.integers(0, 15))
    months_kept = int(rng.geometric(CHURN_RISK[plan])) - 1
    cancelled = signup + months_kept
    if cancelled > TODAY:
        cancelled = None   # cancellation month hasn't arrived yet
    subscribers.append({
        "id": f"SS-{1000 + i}",
        "plan": plan,
        "signup_month": signup,
        "cancelled_month": cancelled,   # None = still subscribed today
        "monthly_fee": PLAN_FEES[plan],
    })

# Founding members: Premium at the £11.99 launch offer, imported from the
# old billing spreadsheet, which had its own ideas about column names.
for i in range(34):
    months_kept = int(rng.geometric(0.018)) - 1
    cancelled = months_kept
    if cancelled > TODAY:
        cancelled = None
    subscribers.append({
        "id": f"SS-{9000 + i}",
        "plan": "Premium",
        "signup_month": 0,
        "cancelled_month": cancelled,
        "monthly_price": 11.99,
    })

# Launch-week trialists: Grabbed the free first month, left before paying.
for i in range(10):
    subscribers.append({
        "id": f"SS-{9500 + i}",
        "plan": "Basic",
        "signup_month": 0,
        "cancelled_month": 0,
        "monthly_fee": PLAN_FEES["Basic"],
    })

rng.shuffle(subscribers)

active = sum(1 for s in subscribers if s["cancelled_month"] is None)
print(f"Created subscribers: {len(subscribers)} records "
      f"({active} active, {len(subscribers) - active} cancelled)")

## 🔍 Stage 2: First look

*(about 15 minutes)*

Your Associate "ran these four cells overnight" to look at the data. **[PREDICT-RUN](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-predict-run)** (see the [AI guide](../../Guides/ai-guide.md)): Write one line in each ✏️ cell *before* you run its code cell. Then run it and note any surprise.

> ✏️ **Your prediction:** *(one line: What will this cell output? Write it BEFORE running.)*
>
> 

In [ ]:
# First look 1: How big is the job, and what does one record look like?
print(len(subscribers))
print(subscribers[0])

> ✏️ **Your prediction:** *(one line: What will this cell output? Write it BEFORE running.)*
>
> 

In [ ]:
# First look 2: How does the base split across plans? (dict comprehension)
plan_counts = {plan: sum(1 for s in subscribers if s["plan"] == plan)
               for plan in ["Basic", "Standard", "Premium"]}
print(plan_counts)

> ✏️ **Your prediction:** *(one line: What will this cell output? Write it BEFORE running. A ranked list of fees, or…?)*
>
> 

In [ ]:
# First look 3: The three most expensive subscriptions on the books.
# (This cell is EXPECTED to fail. Read the traceback bottom-up.)
priciest = sorted(subscribers, key=lambda s: s["monthly_fee"], reverse=True)
print(priciest[:3])

> ✏️ **Your prediction:** *(one line: What will this cell output? Write it BEFORE running.)*
>
> 

In [ ]:
# First look 4: The Partner said imported data has moods. Survey the keys.
missing_fee = [s for s in subscribers if "monthly_fee" not in s]
print(f"Records without a 'monthly_fee' key: {len(missing_fee)} of {len(subscribers)}")
print(f"Example: {missing_fee[0]}")

**First look findings.** First look 3 failed with a `KeyError: 'monthly_fee'`. First look 4 shows why: **34 records have no `monthly_fee` key at all.** They are the founding members from the old spreadsheet, and they carry their fee under `monthly_price` instead, at the £11.99 launch rate.

Carry two things into Build:

1. Any code that touches **fees** must cope with both fee keys. Use `.get()` with a fallback, or a `try`/`except KeyError` that *handles* <!-- plain-language: everyday abcd record -->the record rather than hides it
2. Churn and lifetime need only `plan`, `signup_month`, and `cancelled_month`. Every record has those three. Bronze and Silver need only those fields, so the missing fee key cannot affect them. The danger is in any calculation that uses fees

First look 3 failed *loudly*. Remember that for Stage 4.

## 🛠️ Stage 3: Build

*(about 40 minutes)*

Note anything worth keeping for the AI Use Record cell as you go.

### 🥉 Bronze

**Deliverable:** A churn table the memo can quote. For each plan, print the number of subscribers, the number who have cancelled, and the churn rate as a percentage to one decimal place.

This is **[GENERATE-JUDGE](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-generate-judge)** (see the [AI guide](../../Guides/ai-guide.md)). Here is a prompt to start from. It already carries what First look taught you:

> *"I have a Python list `subscribers` of dicts with keys `plan` (Basic/Standard/Premium) and `cancelled_month` (a month number, or None if still active). Using list comprehensions, count for each plan the members and the cancellations. Then print a table with the churn rate as a percentage to one decimal place. Compare with `is not None`, not truthiness."*

Then **verify before you ship**:

1. **Reconcile the totals:** The three member counts must sum to 320. The three cancellation counts must sum to the cancelled total the data-setup cell printed
2. **One hand-check:** `subscribers[:20]` is small enough to count by eye. Count the Premium records and the Premium cancellations among them yourself, then run your comprehensions on only that slice and check they agree

A subscriber with `cancelled_month = 0` cancelled at launch, and `0` counts as false in Python. A plain `if s["cancelled_month"]` would miss them. The Debrief has more.

In [ ]:
# 🥉 Bronze: Build it here with your Associate, then verify it.
# TODO: For each plan, find members, cancellations, and churn rate (1 decimal place).
#       Comprehensions encouraged; compare with `is not None`.
# TODO: Then reconcile: Member counts sum to 320, cancellations to the
#       data-setup total.

plans = ["Basic", "Standard", "Premium"]

for plan in plans:
    # members = ...
    # cancelled = ...
    # rate = ...
    pass

**✍️ One sentence for the memo:** *(Which plan is churning worst by rate, and is the "Premium is our weakest plan" rumour true? Careful: One plan has the most cancellations, another the worst rate.)*

> 

### 🥈 Silver

StreamShore's own Associate already answered the second board question (*"how long does a customer stay?"*). Its draft is below. It runs without error and prints a confident number. It uses one piece of Python the Demonstration did not cover: `x if condition else y`. Python calls this a [conditional expression](https://docs.python.org/3/reference/expressions.html#conditional-expressions), a one-line if/else. Read the draft's version aloud: *The end month is the cancellation month if there is one, otherwise today.* **Predict what it will print, then run it.**

In [ ]:
# StreamShore's Associate draft: "average customer lifetime in months".
# It runs. The question is what the number MEANS.
lifetimes = []
for s in subscribers:
    end = s["cancelled_month"] if s["cancelled_month"] is not None else TODAY
    lifetimes.append(end - s["signup_month"])

avg_lifetime = sum(lifetimes) / len(lifetimes)
print(f"Average customer lifetime: {avg_lifetime:.1f} months")

Trace the data flow before you read on. For a cancelled subscriber, `end - signup` is a real, completed lifetime. But 198 subscribers have *not* cancelled, and the draft treats every one of them **as if they cancelled today**. Someone who joined last month counts as "1 month". A loyal founder counts as "17", as though their story is over.

When you point this out, the Associate proposes two fixes. Ask your own Associate for its fixes and compare:

**Fix A: Exclude the actives.** Average only the subscribers who have cancelled. Every number in it is a completed lifetime. It leaves out your most loyal customers, because they have not cancelled. It answers: *"When people quit, how quickly do they quit?"*

**Fix B: Keep everyone, and label honestly.** Keep the draft's arithmetic but report it as months *so far*, a **lower bound**. Write it like this: "Customers have stayed **at least** N months on average, and most are still with us." Statisticians call this censoring: You see only part of a lifetime that is still running. It answers: *"How much loyalty have we observed to date?"*

**Your Silver task:** Judge, don't merely pick.

1. Ask your Associate for the case for each in one prompt: *"Argue for A, then argue for B. Then say which you'd put in a board memo about which plan to push next quarter, and why"*
2. Compute **both** numbers in the cell below, labelled
3. In the markdown cell after it, choose one for the memo and defend the choice **in business terms**. What question does the board need answered, and what would each number make them believe? There is no single right answer. The defence is the deliverable

Things that might sway you:

- StreamShore is 18 months old
- The board is choosing which plan to push
- What happens to Fix A's number in a month when many new customers join?
- What happens to Fix B's number if nobody ever cancelled at all?

In [ ]:
# 🥈 Silver: Compute BOTH candidate numbers, clearly labelled.
# TODO Fix A: Average lifetime across CANCELLED subscribers only.
# TODO Fix B: The draft's average, relabelled as a lower bound
#             ("at least N months, X% still active").

pass

**✍️ My choice and defence:** *(A or B, and why: 3-4 sentences, in business terms, your own reasoning rather than the Associate's transcript)*

> 

### 🥇 Gold

The Partner, reading over your shoulder: *"Good. Now tell them something they don't already suspect."*

**Deliverable:** One open analysis of your own design that adds a decision-relevant sentence to the memo, built by directing your Associate. Ideas you can use or ignore:

- **Revenue retained by cohort:** Group subscribers by signup quarter (months 0-2, 3-5, and so on). For each cohort, compute the total revenue to date and the share still active, coping with both fee keys. Is StreamShore's retention getting better or worse as the company matures?
- **The trial leak:** How many subscribers cancel in their very first month, and which plan do they arrive on? What is that worth per month in lost fees?
- **A `Subscriber` class:** Use the Demonstration's class section as your template. Give `__init__` one dict, and make it cope with both fee keys. Add methods such as `months_active()` and `revenue_to_date()`. Rebuild one earlier number with the class. Then judge: Does the class make the memo code easier to read, or is it extra work for a one-off job?
- **Your own question:** Anything whose answer would change what the board does on Friday. State it in the opening comment, answer it, and verify one number by a second route

Gold has no upper limit and no marking scheme. It ends when you can print the evidence *and* say, in one sentence, what the board should do about it.

In [ ]:
# 🥇 Gold. My question: (state it here in a comment)
# Decision it informs:
# TODO: Design and build with your Associate; verify one number a second way.

pass

## 🚩 Stage 4: Red Flags

*(about 15 minutes)*

**[BUG-HUNT](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-bug-hunt)** (see the [AI guide](../../Guides/ai-guide.md)). While you were busy, StreamShore's Associate "finished" the revenue analysis for Friday's board pack. It runs top to bottom without an error and prints a tidy ranking of the three plans by total revenue collected since launch. The board wants to know which plans deserve next quarter's marketing budget.

Run it. Then audit it. First look told you everything you need.

In [ ]:
# The Associate's board-pack draft: Total revenue collected per plan since launch.
# The Associate: "I added error handling so it copes with the messy legacy records."
revenue_by_plan = {}
processed = []

for s in subscribers:
    try:
        end = s["cancelled_month"] if s["cancelled_month"] is not None else TODAY
        months_paid = end - s["signup_month"]
        revenue_by_plan[s["plan"]] = (revenue_by_plan.get(s["plan"], 0)
                                      + months_paid * s["monthly_fee"])
        processed.append(s["id"])
    except KeyError:
        pass   # skip any incomplete records

print("=== Revenue by plan (since launch) ===")
ranking = sorted(revenue_by_plan.items(), key=lambda kv: kv[1], reverse=True)
for plan, total in ranking:
    print(f"{plan:9s} £{total:,.2f}")

The ranking looks plausible. Standard sits on top and Premium comes second, far behind. That fits the "Premium is our weakest plan" rumour.

**Your audit, three steps:**

1. **Find it.** The `try`/`except KeyError: pass` is not error handling. *Which records does it silently discard, and how many?* Compare `len(processed)` with `len(subscribers)`, the reconciliation habit from Bronze
2. **Explain it in business terms** in the cell below. *Whose* revenue vanished, what does the ranking claim now, and what would Friday's board regret doing on these numbers? The discarded records are not a random sample, so think about who the founding members are and which plan they sit on
3. **Fix it by directing, not typing.** Tell your Associate what is wrong and what "handled" should mean here. For example: *"This draft silently drops every record where the fee sits under `monthly_price` instead of `monthly_fee`. Make it use either key, report how many records used the fallback, and crash loudly if a record has neither."* Paste its corrected version into the cell below, run it, and reconcile: All 320 records must now be counted

**✍️ The bug, in business terms:** *(3-4 sentences: What is silently wrong, and which plan's revenue is understated and why? What wrong <!-- plain-language: everyday decision -->decision would Friday's board take on these numbers?)*

> 

In [ ]:
# 🚩 Fixed board-pack numbers: Corrected by directing your Associate.
# TODO: Paste the corrected version here, run it, and reconcile:
#       all 320 records counted, fallback usage reported.

pass

## 📓 Stage 5: Debrief

*(about 10–15 minutes)*

**[TEACH-BACK](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-teach-back)** (see the [AI guide](../../Guides/ai-guide.md)). Answer all three in the cells below, then copy them into your `QuestLog.md` as a dated entry with the badge you are claiming.

**1. Client explanation:** "Explain this week's key technique to the client in three sentences, no jargon." *(For Quest 02b: Why is there more than one honest answer to "how long does a customer stay?", and how did you choose yours?)*

> 

**2. Associate audit:** "Where was your Associate wrong, misleading, or unhelpful this week? Paste the exchange." *(The lifetime draft and the silently dropped founders both count, but anything from your own chat is better.)*

> 

**3. Quiz me:** "Ask your Associate to set you three questions on this week's material. Paste the quiz and answer it *without* AI help." *(Suggested prompt: "Set me three short questions (easy, medium, hard). Draw them from list comprehensions, sorted with a lambda key, why `except: pass` is dangerous, and what `__init__` and `self` do in a class. Don't show answers until I've tried.")*

> 

### 📔 AI Use Record: Today's highlights

[SpecStory](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-specstory) keeps today's full transcript (see the [Your workstation guide](../../Guides/codespaces-start.md)). Paste the **two** exchanges from today that a marker should see. Add one line of your own commentary to each. This habit becomes your AI Use Record in the assessments ([AI Guide, section 7](../../Guides/ai-guide.md)).

**Best exchange** *(the prompt that worked, and why it worked)*:

> **Me:** …
>
> **Associate:** …
>
> **Why it worked:** …

**Worst exchange** *(where the Associate was wrong, misleading, or useless, and how you caught it)*:

> **Me:** …
>
> **Associate:** …
>
> **How I caught it:** …

## 🚪 Before you leave the office

Choose **Run All**. It should stop only at First look 3, the cell that is meant to fail. Click the cell after it and open the dropdown next to its **Run** button. Choose **Execute Cell and Below** to check that everything else runs without an error. <!-- plain-language: everyday commit -->Commit and push your work. Write the [QuestLog](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-questlog) entry while it is fresh. Read the Debrief *after* you have attempted the tiers. It holds the worked answers and the [AI failure gallery](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-ai-failure-gallery).